# UT1.7 - Limpieza de datos

**Módulo:** Análisis de Datos con Python

**Resultado de Aprendizaje:** RA1 - Selecciona y utiliza las librerías principales de Python para la importación, manipulación, limpieza y exportación de datos.

**Nombre del Alumno/a:** ________________________________________

**Instrucciones:** Completa los bloques `# TU CÓDIGO AQUÍ` y ejecuta la celda de test correspondiente para comprobar tu solución.

In [ ]:
%reset -f
%run notebook_setup.py
%whos

## 1. Limpieza de datos (CE1.e, CE1.f)

En un proyecto real los datos llegan sucios: valores ausentes, duplicados, tipos incorrectos, textos con espacios o mayúsculas mezcladas y fechas imposibles de parsear.
La limpieza es la fase de **transformación** del proceso **ETL**:

| Fase | Pregunta que responde | Herramientas de Pandas |
| :--- | :--- | :--- |
| **E**xtract | ¿De dónde vienen los datos? | `read_csv`, `read_json`, `read_parquet`, `read_excel` |
| **T**ransform | ¿Son correctos y homogéneos? | `dropna`, `fillna`, `drop_duplicates`, `astype`, `to_datetime`, `.str.*`, `.loc[]` |
| **L**oad | ¿Dónde y con qué formato se guardan? | `to_csv`, `to_json`, `to_parquet`, `to_excel` |

Checklist de saneamiento, en orden:

1. **Registros sin dato clave**: se eliminan (`dropna(subset=[...])`), no se imputan.
2. **Nulos en columnas numéricas**: se imputan con media o mediana (`fillna`), según haya outliers.
3. **Duplicados**: `duplicated()` para detectarlos y `drop_duplicates()` para eliminarlos.
4. **Tipos**: `astype`, `pd.to_numeric(errors="coerce")`, `pd.to_datetime(errors="coerce")`.
5. **Textos**: `str.strip()`, `str.lower()`, `str.upper()`, `str.replace()`.
6. **Valores imposibles** (precios negativos, edades de 300 años): se marcan como nulos o se recortan (`clip`).

### 1.1. Técnicas de saneamiento: referencia rápida

| Necesidad | Función | Ejemplo |
| :--- | :--- | :--- |
| Ver nulos por columna | `isna().sum()` | `df.isna().sum()` |
| Eliminar filas con nulos | `dropna()` | `df.dropna(subset=["email"])` |
| Imputar nulos | `fillna()` | `df["edad"].fillna(df["edad"].median())` |
| Contar duplicados | `duplicated()` | `df.duplicated(subset=["codigo"]).sum()` |
| Eliminar duplicados | `drop_duplicates()` | `df.drop_duplicates(subset=["codigo"], keep="first")` |
| Texto a número | `to_numeric()` | `pd.to_numeric(df["precio"], errors="coerce")` |
| Texto a fecha | `to_datetime()` | `pd.to_datetime(df["alta"], errors="coerce")` |
| Limpiar cadenas | `.str.*` | `df["email"].str.strip().str.lower()` |
| Recortar outliers | `clip()` | `df["edad"].clip(lower=0, upper=120)` |

### 1.2. Ejemplo resuelto: de datos sucios a tabla limpia

El patrón habitual es encadenar transformaciones: filtrar el dato clave, normalizar texto, imputar, convertir tipos y deduplicar.

In [ ]:
import numpy as np
import pandas as pd

df_sucio = pd.DataFrame({
    "email": ["  ANA@MAIL.COM ", None, " luis@mail.com", "ana@mail.com"],
    "edad": [25.0, np.nan, 41.0, 25.0],
    "alta": ["2023-01-15", "2023-02-30", "2023-03-01", "2023-01-15"],
})

df_limpio = (
    df_sucio
    .dropna(subset=["email"])                                  # 1. fuera registros sin dato clave
    .assign(
        email=lambda d: d["email"].str.strip().str.lower(),    # 2. normalizar texto
        edad=lambda d: d["edad"].fillna(d["edad"].median()),  # 3. imputar nulos
        alta=lambda d: pd.to_datetime(d["alta"], errors="coerce"),  # 4. convertir tipos
    )
    .drop_duplicates()                                          # 5. deduplicar
)

print(df_limpio)
print("\nTipos:\n", df_limpio.dtypes)
print("\nNulos por columna:\n", df_limpio.isna().sum())

### 1.3. Ejercicios

Los ejercicios usan `pd` y `np`, importados en el apartado 1.2.

#### Ejercicio 1.1: Imputación de Nulos y Parsing de Fechas (CE1.f)
Implementa `limpiar_usuarios(df)` que:
1. Elimine los registros con `nombre` nulo.
2. Impute `edad` con la mediana.
3. Convierta `fecha_registro` a `datetime` con `errors="coerce"`.

In [ ]:
def limpiar_usuarios(df: pd.DataFrame) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función limpiar_usuarios")

In [ ]:
# TEST 1.1
df_dirty = pd.DataFrame({
    "nombre": ["Alicia", None, "Carlos", "Diana"],
    "edad": [25.0, 40.0, np.nan, 35.0],
    "fecha_registro": ["2023-01-01", "2023-02-01", "invalida", "2023-04-15"]
})

res_11 = limpiar_usuarios(df_dirty)
assert len(res_11) == 3
assert res_11.loc[res_11["nombre"] == "Carlos", "edad"].values[0] == 30.0
assert pd.isna(res_11.loc[res_11["nombre"] == "Carlos", "fecha_registro"].values[0])
print("✅ Test Ejercicio 1.1 superado.")

#### Ejercicio 1.2: Sanitización de Cadenas y Deduplicación (CE1.f)
Implementa `normalizar_catalogo(df)` que:
1. Quite los espacios de `codigo` y de `producto`.
2. Pase `producto` a MAYÚSCULAS.
3. Elimine los duplicados por `codigo`, conservando el primer registro.

In [ ]:
def normalizar_catalogo(df: pd.DataFrame) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función normalizar_catalogo")

In [ ]:
# TEST 1.2
df_cat_dirty = pd.DataFrame({
    "codigo": ["  A10 ", "A10", " B20"],
    "producto": ["  laptop ", "LAPTOP", "teclado  "]
})

res_12 = normalizar_catalogo(df_cat_dirty)
assert len(res_12) == 2
assert res_12.loc[res_12["codigo"] == "A10", "producto"].values[0] == "LAPTOP"
assert res_12.loc[res_12["codigo"] == "B20", "producto"].values[0] == "TECLADO"
print("✅ Test Ejercicio 1.2 superado.")

#### Ejercicio 1.3: Pipeline ETL Clean (CE1.e, CE1.f)
Implementa `pipeline_limpieza_ventas(df)` que:
1. Elimine los registros con `email` nulo o vacío.
2. Normalice `email` (sin espacios y en minúsculas).
3. Convierta `precio` y `cantidad` a numérico, descartando los valores negativos o no convertibles.
4. Calcule la columna `total = precio * cantidad`.

In [ ]:
def pipeline_limpieza_ventas(df: pd.DataFrame) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función pipeline_limpieza_ventas")

In [ ]:
# TEST 1.3
df_raw = pd.DataFrame({
    "email": ["  USER1@TEST.COM ", None, "", "user2@test.com", "user3@test.com"],
    "precio": ["10.5", "20.0", "15.0", "-5.0", "invalid"],
    "cantidad": ["2", "1", "3", "4", "2"]
})

res_13 = pipeline_limpieza_ventas(df_raw)
assert len(res_13) == 1
assert res_13.iloc[0]["email"] == "user1@test.com"
assert res_13.iloc[0]["total"] == 21.0
print("✅ Test Ejercicio 1.3 superado.")